# Sprint 2: Attention U-Net baseline (Kibera)

Trains the Attention U-Net flood head on the tiles produced in Sprint 1 and checks whether it converges
(proposal Sprint 2 exit criterion: validation IoU >= 0.60).

This notebook is self-contained and restart-safe. It clones the repository fresh, finds the Sprint 1
raster stack in Drive, rebuilds the tiles from it, and trains. It applies **no patch cells**: every
Sprint 1 fix must already be committed to GitHub.

**Three things to keep in mind when you read the results**

1. **The label is a proxy, not observed flooding.** `flood_label` comes from a thresholded Sentinel-1 backscatter
   change. An IoU of 0.60 means the model agrees with that proxy 60%, which says nothing yet about real flood
   extent. Validate against the NRRP high-risk zones before quoting this as flood-prediction accuracy.
2. **The validation design differs from the proposal.** The proposal specifies 5-fold spatially contiguous
   cross-validation. A single Kibera export yields only a few distinct 256x256 locations, so this notebook holds
   out one whole location instead. This should be stated as a deliberate amendment in the write-up.
3. **Transfer learning only happens if you supply a compatible checkpoint.** With no checkpoint the network trains
   from scratch, and the notebook says so in its output.

## 0. Before you run

- Runtime > Change runtime type > **T4 GPU** (do this first, it restarts the session).
- The Sprint 1 raster export must exist in Drive: `MyDrive/nairobi_risk_mapping/kibera_raster_stack*.tif`.
- The Sprint 2 code (`src/models/`, including `evaluation.py`) must be pushed to GitHub. The next cells check this.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. This dataset is small enough to train on CPU, but a T4 is much faster.")

In [ ]:
%%bash
# One checkout for Sprints 1–6. This branch carries the Overpass client
# (project User-Agent; kumi, the French whitelist, and the Swiss extract
# are not queried). After that change is on main, set BRANCH=main.
BRANCH=cursor/overpass-identity-backoff-b69d
REPO=https://github.com/emmanuelamor5/nairobi-risk-mapping.git
cd /content
if [ ! -d nairobi-risk-mapping/.git ]; then
  git clone "$REPO" nairobi-risk-mapping
fi
cd nairobi-risk-mapping
git fetch origin "$BRANCH" || git fetch origin main
if git rev-parse --verify --quiet "origin/$BRANCH" >/dev/null; then
  git checkout "$BRANCH"
  git pull --ff-only origin "$BRANCH"
else
  echo "Branch $BRANCH is not on origin; using main."
  git checkout main
  git pull --ff-only origin main
fi
git rev-parse --abbrev-ref HEAD
git log -1 --oneline
grep -n OVERPASS_USER_AGENT src/config.py | head -3


In [ ]:
%cd /content/nairobi-risk-mapping

from pathlib import Path
required = [
    "src/config.py", "src/data_pipeline.py", "src/utils/gee_utils.py", "src/utils/graph_utils.py",
    "src/models/attention_unet.py", "src/models/attention_unet_model.py", "src/models/dataset.py",
    "src/models/losses.py", "src/models/metrics.py", "src/models/transfer_learning.py",
    "src/models/evaluation.py",
]
missing = [p for p in required if not Path(p).exists()]
assert not missing, (
    f"Not in the cloned repo: {missing}. Commit and push the Sprint 2 files from your machine, "
    "then rerun from the top of this notebook."
)
print("Repository check passed.")

In [ ]:
# DataPipeline (used below only for tiling) imports the Sprint 1 utilities, so their dependencies are needed.
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import logging, random
import numpy as np
import torch

logging.basicConfig(level=logging.INFO, force=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

## 1. Locate and verify the Sprint 1 raster stack

Earth Engine never overwrites an export, so repeated runs leave `kibera_raster_stack(1).tif`, `(2).tif` and so on
in Drive. This cell lists every candidate and uses the newest, so you can see which file is actually being used.

In [ ]:
import glob, os, time

DRIVE_DIR = "/content/drive/MyDrive/nairobi_risk_mapping"
candidates = sorted(glob.glob(f"{DRIVE_DIR}/kibera_raster_stack*.tif"), key=os.path.getmtime, reverse=True)
assert candidates, f"No kibera_raster_stack*.tif found in {DRIVE_DIR}. Run the Sprint 1 raster export first."

for p in candidates:
    print(f"{os.path.basename(p):38s} {os.path.getsize(p)/1e6:7.1f} MB   modified {time.ctime(os.path.getmtime(p))}")
if len(candidates) > 1:
    print("\nMore than one export found; using the newest. Older copies may come from earlier flood-label "
          "versions. Delete them once you are sure which one is current.")

stack_path = candidates[0]
print("\nUsing:", stack_path)

In [ ]:
import rasterio

BAND_NAMES = ["B2", "B3", "B4", "B8", "B11", "B12", "flood_label", "elevation", "slope", "twi"]

with rasterio.open(stack_path) as src:
    data = src.read()
    print("Dimensions (height x width):", src.height, "x", src.width, "| bands:", src.count, "| CRS:", src.crs)

assert data.shape[0] == len(BAND_NAMES), f"Expected {len(BAND_NAMES)} bands, found {data.shape[0]}"
assert min(data.shape[1:]) >= 256, "Raster is smaller than one 256px tile; widen the export buffer in Sprint 1."

print("\nPer-band NaN fraction:")
for name, band in zip(BAND_NAMES, data):
    print(f"  {name:12s} {np.isnan(band).mean():.4f}")

label = data[6]
finite = label[np.isfinite(label)]
print("\nflood_label unique values:", np.unique(finite).tolist())
print(f"flood_label flooded fraction over the whole raster: {finite.mean():.3f}")
assert set(np.unique(finite).tolist()) <= {0.0, 1.0}, "flood_label is not binary"
assert 0.01 < finite.mean() < 0.99, "flood_label is nearly constant; the model cannot learn from it"

## 2. Rebuild the tiles

Tiles live on the Colab disk and are not committed to Git, so they are regenerated from the Drive raster on each run.
This takes seconds and is deterministic. Earlier tile files are deleted first so the manifest cannot point at stale ones.

In [ ]:
import shutil
from src.data_pipeline import DataPipeline

shutil.rmtree("data/tiles/kibera", ignore_errors=True)
pipeline = DataPipeline(settlement_key="kibera")   # no Earth Engine project needed for tiling
manifest_path = str(pipeline.tile_raster_stack(stack_path, split="train"))
print("Manifest:", manifest_path)

In [ ]:
import json

with open(manifest_path) as f:
    manifest = json.load(f)

locations = sorted({(t["row_off"], t["col_off"]) for t in manifest})
print(f"{len(manifest)} tiles at {len(locations)} distinct spatial locations: {locations}")
assert len(locations) >= 2, (
    "Need at least 2 distinct locations to hold one out. Widen the Sprint 1 export buffer, "
    "or check how many windows were discarded for low valid-data fraction."
)

def orig_entry(loc):
    return next(t for t in manifest if (t["row_off"], t["col_off"]) == loc and t["tile_id"].endswith("_orig"))

print()
for loc in locations:
    e = orig_entry(loc)
    lab = np.load(e["path"])[6]
    vals = np.unique(lab[np.isfinite(lab)]).tolist()
    frac = float(np.nanmean(lab))
    flag = ""
    if vals != [0.0, 1.0]:
        flag = "  <-- label is not a 0/1 mix"
    elif frac < 0.02 or frac > 0.98:
        flag = "  <-- near-constant label"
    print(f"location {loc}: label values {vals}, flooded {frac*100:5.1f}%, "
          f"NaN {np.isnan(lab).mean()*100:.2f}%, valid-data fraction {e['valid_fraction']:.2f}{flag}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(locations), 2, figsize=(10, 5 * len(locations)), squeeze=False)
for i, loc in enumerate(locations):
    tile = np.load(orig_entry(loc)["path"])
    rgb = np.nan_to_num(np.stack([tile[2], tile[1], tile[0]], axis=-1))   # B4, B3, B2
    axes[i, 0].imshow(rgb);                            axes[i, 0].set_title(f"RGB, location {loc}")
    axes[i, 1].imshow(tile[6], cmap="Blues", vmin=0, vmax=1)
    axes[i, 1].set_title(f"flood label ({np.nanmean(tile[6])*100:.1f}% flooded)")
plt.tight_layout(); plt.show()

## 3. Train/validation split, and the overlap problem

Tiles are 256px wide but placed 206px apart, so neighbouring locations share a 50px strip of the same ground.
If one location is held out for validation, part of it is still visible in a training tile, which inflates the
validation score. The cell below measures that overlap, and section 7 re-evaluates with the shared pixels excluded.

In [ ]:
from src.models.dataset import spatial_holdout_split
from src.models.evaluation import validation_overlap_mask

split = spatial_holdout_split(manifest_path)
val_loc = split["holdout_location"]
train_locs = sorted({(t["row_off"], t["col_off"]) for t in split["train"]})
print(f"Train: {len(split['train'])} tiles at {train_locs}")
print(f"Val:   {len(split['val'])} tiles at {val_loc}")

safe_mask = validation_overlap_mask(val_loc, train_locs, tile_size=256)
print(f"\nShare of the validation tile also covered by a training tile: {(~safe_mask).mean()*100:.1f}%")
print(f"Pixels usable for leakage-free evaluation: {int(safe_mask.sum())} of {safe_mask.size}")
assert safe_mask.sum() > 0, "Every validation pixel overlaps a training tile; no clean evaluation is possible."

## 4. Model

`PRETRAINED_CHECKPOINT_PATH = None` trains from scratch. To use transfer learning, set it to a `.pt` file saved from
**this same `AttentionUNet` architecture**. The loader matches layers by exact name and shape, so a checkpoint from a
differently built network (for example a ResNet-backed U-Net) matches nothing, and the loader raises an error rather
than quietly leaving you with a random encoder. The 9-channel input layer can never transfer from a 2-channel
Sentinel-1 model; it is reinitialised and kept trainable together with the first encoder stage.

In [ ]:
PRETRAINED_CHECKPOINT_PATH = None   # e.g. "/content/drive/MyDrive/nairobi_risk_mapping/pretrained_encoder.pt"

from src.models.attention_unet_model import AttentionUNetModel
from src.models.transfer_learning import trainable_parameter_count

model = AttentionUNetModel(
    in_channels=9,
    base_channels=32,
    pretrained_checkpoint=PRETRAINED_CHECKPOINT_PATH,
    freeze_pretrained_encoder=True,
)

if model.transfer_report is not None:
    r = model.transfer_report
    print(f"Transferred {len(r['transferred'])} tensors; reinitialised: {r['reinitialised']}")
else:
    print("Training from scratch: this is NOT the transfer-learning setup described in the proposal.")
print(trainable_parameter_count(model.network))

## 5. Baseline to beat

A model that predicts "flooded" everywhere gets an IoU equal to the flooded fraction. The 0.60 target is only
meaningful relative to that number.

In [ ]:
val_orig = next(t for t in split["val"] if t["tile_id"].endswith("_orig"))
val_label = np.nan_to_num(np.load(val_orig["path"])[6])
baseline_iou = float((val_label[safe_mask] >= 0.5).mean())
print(f"IoU of 'flood everywhere' on the leakage-free validation pixels: {baseline_iou:.3f}")

## 6. Train

Adam, initial learning rate 1e-3, cosine annealing, BCE + Dice loss, as specified in the proposal. The best
checkpoint is saved to Drive by validation IoU.

In [ ]:
CHECKPOINT_DIR = f"{DRIVE_DIR}/checkpoints"

result = model.fit(
    train_manifest_entries=split["train"],
    val_manifest_entries=split["val"],
    epochs=50,
    batch_size=2,
    learning_rate=1e-3,
    checkpoint_dir=CHECKPOINT_DIR,
)
print(f"\nBest validation IoU during training (includes the shared strip): {result['best_val_iou']:.4f}")

In [ ]:
history = result["history"]
epochs = [h["epoch"] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs, [h["train_loss"] for h in history], label="train")
axes[0].plot(epochs, [h["val_loss"] for h in history], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(epochs, [h["train_iou"] for h in history], label="train IoU")
axes[1].plot(epochs, [h["val_iou"] for h in history], label="val IoU")
axes[1].axhline(0.60, color="red", linestyle="--", label="target 0.60")
axes[1].axhline(baseline_iou, color="grey", linestyle=":", label="flood-everywhere baseline")
axes[1].set_title("IoU"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.show()

## 7. Leakage-free evaluation

Reloads the best checkpoint and scores it only on validation pixels that no training tile covers. Note that the
checkpoint itself was chosen using this same validation location, so the number is still somewhat optimistic. That is
acceptable for a convergence check but not for a final result.

In [ ]:
from src.models.evaluation import evaluate_masked

model.load_checkpoint(f"{CHECKPOINT_DIR}/attention_unet_best.pt")
clean = evaluate_masked(model, val_orig["path"], safe_mask)

print(f"During training (includes shared strip): IoU {result['best_val_iou']:.3f}")
print(f"Leakage-free:                            IoU {clean['iou']:.3f}   F1 {clean['f1']:.3f}")
print(f"Flood-everywhere baseline:               IoU {clean['all_flooded_baseline_iou']:.3f}")
print(f"Pixels evaluated: {clean['evaluated_pixels']} of {clean['total_pixels']}")
print("\nSprint 2 convergence check (IoU >= 0.60):", "MET" if clean["iou"] >= 0.60 else "NOT MET")

In [ ]:
tile = np.load(val_orig["path"])
from src.models.dataset import INPUT_BAND_INDICES
inputs = torch.from_numpy(np.nan_to_num(tile[INPUT_BAND_INDICES], nan=0.0)).float().unsqueeze(0)
proba = model.predict_proba(inputs)[0, 0].numpy()

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
axes[0].imshow(np.nan_to_num(np.stack([tile[2], tile[1], tile[0]], axis=-1))); axes[0].set_title("RGB")
axes[1].imshow(np.nan_to_num(tile[6]), cmap="Blues", vmin=0, vmax=1);          axes[1].set_title("flood label (proxy)")
axes[2].imshow(proba, cmap="Blues", vmin=0, vmax=1);                           axes[2].set_title("predicted probability")
axes[3].imshow(safe_mask, cmap="gray");                                         axes[3].set_title("white = evaluated")
plt.show()

In [ ]:
summary = {
    "best_val_iou_during_training": result["best_val_iou"],
    "leakage_free": clean,
    "trained_from_scratch": PRETRAINED_CHECKPOINT_PATH is None,
    "train_locations": train_locs,
    "val_location": list(val_loc),
    "raster_used": os.path.basename(stack_path),
    "seed": SEED,
}
with open(f"{CHECKPOINT_DIR}/sprint2_summary.json", "w") as f:
    json.dump({"summary": summary, "history": history}, f, indent=2, default=str)
print("Saved summary and history to", CHECKPOINT_DIR)

## Sprint 2 exit checklist

- [ ] Leakage-free IoU >= 0.60 **and** clearly above the flood-everywhere baseline
- [ ] Results labelled as trained from scratch or transferred, whichever is true
- [ ] Spatial-holdout split documented as an amendment to the proposal's 5-fold design
- [ ] `attention_unet_best.pt` and `sprint2_summary.json` present in Drive
- [ ] Flood label validated against NRRP high-risk zones (still open from Sprint 1)

Next: Sprint 3, the GraphSAGE structural-deprivation head on the pruned Kibera graph.